# Analysis of Misclassifications

In [2]:
from pathlib import Path
import json
import pickle

import pandas as pd

## Compute Misclassifications

### Experiment configuration

In [3]:
# # DARPA2000

# dataset = "darpa2000"
# scenario = "s1_inside_dmz"

# logic_file = "darpa_test"

# train_mode = "scratch" 
# train_mode = "pretrained"

# dataset_variant = "down" 
# dataset_variant = "original" 
# dataset_variant = "resampled"

# window_size = 10

# run_id = "20260402_123738"

# experiment_name = f"{logic_file}_{train_mode}_{dataset_variant}_w{window_size}"
# cache_file_name = f"{logic_file}_{dataset_variant}_w{window_size}_test.pkl"

In [4]:
# AIT Log v2

dataset = "aitv2"
scenario = "fox"
# scenario = "santos"

logic_file = "ait_neg"

train_mode = "scratch" 
# train_mode = "pretrained" 

fraction = 25
window_size = 10
run_id = "20260409_165607"

experiment_name = f"{logic_file}_{train_mode}_{fraction}data_w{window_size}"
cache_file_name = f"{logic_file}_{fraction}data_w{window_size}_test.pkl"

### Load Original Flows

In [5]:
scenario_parts = scenario.split("_")
if len(scenario_parts) == 3:
    test_set_tag = f"{scenario_parts[0]}_{scenario_parts[2]}"
else:
    test_set_tag = scenario

In [6]:
df = pd.read_csv(
    f"../data/interim/{dataset}/{test_set_tag}/flows_labeled/all_flows_labeled.csv"
)

df = df.sort_values("start_time").reset_index(drop=True)
df['t_rel'] = df['start_time'] - df['start_time'].min()

### Load DPL Dataset

In [7]:
def load_dpl_dataset(logic_file, cache_file_name):
    dpl_dataset_dir = Path(f"../experiments/{dataset}/{scenario}/deepproblog/{logic_file}/cache/")
    cache_file_test = dpl_dataset_dir / cache_file_name

    cache = pickle.load(open(cache_file_test, "rb"))
    cache_df = pd.DataFrame(cache)
    cache_df.head()

    return cache_df

In [8]:
cache_df = load_dpl_dataset(logic_file, cache_file_name)

### Map Misclassifications to Original Flows

In [9]:
errors_dir = f"../experiments/{dataset}/{scenario}/deepproblog/{logic_file}/errors"
errors_file = (
    f"{errors_dir}/"
    f"{experiment_name}_{run_id}.json"
)
    
with open(errors_file) as f:
    errors = json.load(f)

In [10]:
dpl_to_orig = dict(zip(cache_df['dpl_index'], cache_df['orig_index']))

original_indices = []
mis_y_preds = []
mis_y_trues = []

phase_map = {
    "benign": 0,
    "phase1": 1,
    "phase2": 2,
    "phase3": 3,
    "phase4": 4,
    "phase5": 5,
}

for error in errors:
    dpl_index = error['index']

    original_indices.append(dpl_to_orig[dpl_index])
    y_pred = error['predicted']
    y_true = error['actual']
    mis_y_preds.append(phase_map[y_pred])
    mis_y_trues.append(phase_map[y_true])

mis_df = df.loc[original_indices].copy()
mis_df['y_pred'] = mis_y_preds
mis_df['y_true'] = mis_y_trues

In [11]:
mis_df

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase,t_rel,y_pred,y_true
30,f5137,1.642205e+09,1.642205e+09,0.005200,10.35.33.111,39682,10.35.32.1,53,udp,dns,...,-,17,benign DNS,internal_share,2022-01-15 00:00:20.149405956,2022-01-15 00:00:20.154606104,0,19.280036,1,0
37,f5138,1.642205e+09,1.642205e+09,0.004648,10.35.33.111,46770,10.35.32.1,53,udp,dns,...,-,17,benign DNS,internal_share,2022-01-15 00:00:34.199841022,2022-01-15 00:00:34.204488993,0,33.330471,1,0
2406,f5802,1.642212e+09,1.642212e+09,0.003489,10.35.33.111,43026,10.35.32.1,53,udp,dns,...,-,17,benign DNS,internal_share,2022-01-15 01:53:29.551275015,2022-01-15 01:53:29.554764032,0,6808.681905,2,0
3942,f6246,1.642216e+09,1.642216e+09,0.004885,10.35.33.111,41179,10.35.32.1,53,udp,dns,...,-,17,benign DNS,internal_share,2022-01-15 03:07:38.351530075,2022-01-15 03:07:38.356415033,0,11257.482160,2,0
8674,f16810,1.642228e+09,1.642228e+09,0.004123,192.168.128.4,46199,192.168.255.254,53,udp,dns,...,-,17,benign DNS,inet_firewall,2022-01-15 06:27:53.909753084,2022-01-15 06:27:53.913876057,0,23273.040383,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
470444,f44910,1.642630e+09,1.642630e+09,5.173146,172.17.130.37,59676,172.17.131.81,443,tcp,ssl,...,-,6,proxy,mail,2022-01-19 22:11:44.248986959,2022-01-19 22:11:49.422132969,0,425503.379617,4,0
470445,f85659,1.642630e+09,1.642630e+09,5.172973,172.17.130.37,59676,172.17.131.81,443,tcp,ssl,...,-,6,proxy,webserver,2022-01-19 22:11:44.249099016,2022-01-19 22:11:49.422071934,0,425503.379729,4,0
470447,f85660,1.642630e+09,1.642630e+09,5.169627,172.17.130.37,59678,172.17.131.81,443,tcp,ssl,...,-,6,proxy,webserver,2022-01-19 22:11:44.252429962,2022-01-19 22:11:49.422056913,0,425503.383060,4,0
470448,f44912,1.642630e+09,1.642630e+09,5.032542,172.17.130.37,59680,172.17.131.81,443,tcp,ssl,...,-,6,proxy,mail,2022-01-19 22:11:44.252965926,2022-01-19 22:11:49.285507917,0,425503.383596,4,0


## Misclassification Analysis

### Helper functions

In [12]:
def false_positives(df, phase):
    return df[(df["y_true"] != phase) & (df["y_pred"] == phase)]

def false_negatives(df, phase):
    return df[(df["y_true"] == phase) & (df["y_pred"] != phase)]

def true_positives(df, phase):
    return df[(df["y_true"] == phase) & (df["y_pred"] == phase)]

### Per-Phase Misclassifications

#### Phase 2

In [13]:
df_fp_2 = false_positives(mis_df, 2)
df_fp_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase,t_rel,y_pred,y_true
2406,f5802,1.642212e+09,1.642212e+09,0.003489,10.35.33.111,43026,10.35.32.1,53,udp,dns,...,-,17,benign DNS,internal_share,2022-01-15 01:53:29.551275015,2022-01-15 01:53:29.554764032,0,6808.681905,2,0
3942,f6246,1.642216e+09,1.642216e+09,0.004885,10.35.33.111,41179,10.35.32.1,53,udp,dns,...,-,17,benign DNS,internal_share,2022-01-15 03:07:38.351530075,2022-01-15 03:07:38.356415033,0,11257.482160,2,0
8674,f16810,1.642228e+09,1.642228e+09,0.004123,192.168.128.4,46199,192.168.255.254,53,udp,dns,...,-,17,benign DNS,inet_firewall,2022-01-15 06:27:53.909753084,2022-01-15 06:27:53.913876057,0,23273.040383,2,0
20435,f14838,1.642239e+09,1.642239e+09,30.792696,172.17.130.196,47534,172.17.130.37,443,tcp,ssl,...,-,6,HTTPS,webserver,2022-01-15 09:25:39.342952967,2022-01-15 09:26:10.135648966,0,33938.473583,2,0
24305,f15502,1.642241e+09,1.642241e+09,5.765148,172.17.130.196,54738,172.17.130.37,443,tcp,ssl,...,-,6,HTTPS,webserver,2022-01-15 10:01:29.270170927,2022-01-15 10:01:35.035318851,0,36088.400801,2,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
237206,f32922,1.642507e+09,1.642507e+09,0.000000,10.35.32.1,68,10.35.32.2,67,udp,dhcp,...,-,17,benign,internal_share,2022-01-18 11:59:09.635035038,2022-01-18 11:59:09.635035038,0,302348.765665,2,0
237209,f35994,1.642507e+09,1.642507e+09,14.999674,172.17.130.196,59454,10.35.35.206,443,tcp,ssl,...,-,6,HTTPS,vpn,2022-01-18 11:59:10.918761015,2022-01-18 11:59:25.918435097,0,302350.049391,2,0
237212,f165594,1.642507e+09,1.642507e+09,0.000732,192.168.130.77,3,192.168.130.77,3,icmp,-,...,-,1,benign,attacker0,2022-01-18 11:59:11.628176928,2022-01-18 11:59:11.628908873,0,302350.758807,2,0
237769,f146452,1.642507e+09,1.642507e+09,0.001809,10.9.0.18,43394,172.17.128.1,3306,tcp,-,...,-,6,benign,attacker0,2022-01-18 11:59:14.512196063,2022-01-18 11:59:14.514004946,0,302353.642826,2,0


In [14]:
df_fp_2_protocols = df_fp_2["proto"].value_counts()
print("False Positives for Phase 2 Protocol Distribution:")
print(df_fp_2_protocols)

False Positives for Phase 2 Protocol Distribution:
proto
tcp     50
udp     21
icmp     1
Name: count, dtype: int64


In [15]:
df_fn_2 = false_negatives(mis_df, 2)
df_fn_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase,t_rel,y_pred,y_true
269490,f50590,1.642507e+09,1.642507e+09,0.000000,172.17.130.196,40430,10.35.42.80,443,tcp,-,...,-,6,host_discover_local,vpn,2022-01-18 11:59:51.362143993,2022-01-18 11:59:51.362143993,2,302390.492774,0,2
274431,f53057,1.642507e+09,1.642507e+09,0.000000,172.17.130.196,39508,10.35.57.39,80,tcp,-,...,-,6,host_discover_local,vpn,2022-01-18 12:00:02.168390036,2022-01-18 12:00:02.168390036,2,302401.299020,0,2
274455,f53068,1.642507e+09,1.642507e+09,0.000000,172.17.130.196,46670,10.35.62.40,80,tcp,-,...,-,6,host_discover_local,vpn,2022-01-18 12:00:02.270904064,2022-01-18 12:00:02.270904064,2,302401.401534,0,2
274476,f53077,1.642507e+09,1.642507e+09,0.000000,172.17.130.196,60280,10.35.49.42,80,tcp,-,...,-,6,host_discover_local,vpn,2022-01-18 12:00:02.372315884,2022-01-18 12:00:02.372315884,2,302401.502946,0,2
274498,f53087,1.642507e+09,1.642507e+09,0.000000,172.17.130.196,35034,10.35.53.43,80,tcp,-,...,-,6,host_discover_local,vpn,2022-01-18 12:00:02.474066019,2022-01-18 12:00:02.474066019,2,302401.604696,0,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
384218,f17084,1.642509e+09,1.642509e+09,0.256739,172.17.130.196,34906,10.35.35.206,443,tcp,ssl,...,-,6,dirb_scan,intranet_server,2022-01-18 12:37:13.621602058,2022-01-18 12:37:13.878340960,2,304632.752232,0,2
384392,f78413,1.642509e+09,1.642509e+09,0.273334,172.17.130.196,35022,10.35.35.206,443,tcp,ssl,...,-,6,dirb_scan,vpn,2022-01-18 12:37:30.702065945,2022-01-18 12:37:30.975399971,2,304649.832696,0,2
384436,f17157,1.642509e+09,1.642509e+09,0.256553,172.17.130.196,35050,10.35.35.206,443,tcp,ssl,...,-,6,dirb_scan,intranet_server,2022-01-18 12:37:34.448327065,2022-01-18 12:37:34.704879999,2,304653.578957,0,2
384511,f78451,1.642509e+09,1.642509e+09,0.260237,172.17.130.196,35098,10.35.35.206,443,tcp,ssl,...,-,6,dirb_scan,vpn,2022-01-18 12:37:41.850581884,2022-01-18 12:37:42.110818863,2,304660.981212,0,2


In [16]:
df_fn_2["local_orig"]
df_fn_2["local_resp"]

269490    T
274431    T
274455    T
274476    T
274498    T
         ..
384218    T
384392    T
384436    T
384511    T
384518    T
Name: local_resp, Length: 522, dtype: object

In [17]:
df_tp_2 = true_positives(mis_df, 2)
df_tp_2

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase,t_rel,y_pred,y_true


#### Phase 3

In [18]:
df_fn_3 = false_negatives(mis_df, 3)
df_fn_3

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase,t_rel,y_pred,y_true


In [19]:
df_fp_3 = false_positives(mis_df, 3)
df_fp_3

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase,t_rel,y_pred,y_true
326460,f201292,1.642508e+09,1.642508e+09,3.036930,10.9.0.18,56606,172.17.158.26,443,tcp,-,...,-,6,benign,attacker0,2022-01-18 12:06:40.820636988,2022-01-18 12:06:43.857567072,0,302799.951267,3,0
327586,f202263,1.642508e+09,1.642508e+09,3.035870,10.9.0.18,43100,172.17.148.104,443,tcp,-,...,-,6,benign,attacker0,2022-01-18 12:07:05.877656937,2022-01-18 12:07:08.913527012,0,302825.008287,3,0
327588,f202265,1.642508e+09,1.642508e+09,3.035694,10.9.0.18,34176,172.17.150.104,443,tcp,-,...,-,6,benign,attacker0,2022-01-18 12:07:05.877856016,2022-01-18 12:07:08.913549900,0,302825.008486,3,0
327620,f202289,1.642508e+09,1.642508e+09,3.037510,10.9.0.18,42418,172.17.148.118,443,tcp,-,...,-,6,benign,attacker0,2022-01-18 12:07:05.908005953,2022-01-18 12:07:08.945516109,0,302825.038636,3,0
361713,f229035,1.642508e+09,1.642508e+09,6.034444,10.9.0.18,44746,172.17.130.37,443,tcp,ssl,...,-,6,benign,attacker0,2022-01-18 12:17:21.694422007,2022-01-18 12:17:27.728866100,0,303440.825052,3,0
371416,f233353,1.642508e+09,1.642508e+09,0.104699,10.9.0.18,55202,10.35.35.206,443,tcp,ssl,...,-,6,benign,attacker0,2022-01-18 12:17:43.860316038,2022-01-18 12:17:43.965014935,0,303462.990946,3,0
371454,f233427,1.642508e+09,1.642508e+09,21.082823,10.9.0.18,55210,10.35.35.206,443,tcp,ssl,...,-,6,benign,attacker0,2022-01-18 12:17:55.475393057,2022-01-18 12:18:16.558216095,0,303474.606023,3,0
371523,f233369,1.642508e+09,1.642508e+09,1.966785,10.9.0.18,55216,10.35.35.206,443,tcp,ssl,...,-,6,benign,attacker0,2022-01-18 12:17:58.104749918,2022-01-18 12:18:00.071534872,0,303477.235380,3,0
372971,f233819,1.642508e+09,1.642508e+09,0.287730,10.9.0.18,56078,10.35.35.206,443,tcp,ssl,...,-,6,benign,attacker0,2022-01-18 12:20:20.116074085,2022-01-18 12:20:20.403804064,0,303619.246704,3,0
373747,f234073,1.642508e+09,1.642508e+09,0.267876,10.9.0.18,56584,10.35.35.206,443,tcp,ssl,...,-,6,benign,attacker0,2022-01-18 12:21:32.978652000,2022-01-18 12:21:33.246527910,0,303692.109282,3,0


#### Phase 4

In [20]:
df_fn_4 = false_negatives(mis_df, 4)
df_fn_4

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase,t_rel,y_pred,y_true
384685,f78505,1.642510e+09,1.642510e+09,64.195314,172.17.130.196,51940,172.17.130.37,443,tcp,ssl,...,-,6,online_cracking,vpn,2022-01-18 12:38:41.709909916,2022-01-18 12:39:45.905223846,4,304720.840540,0,4
384741,f78507,1.642510e+09,1.642510e+09,96.252340,172.17.130.196,59296,10.35.35.206,80,tcp,http,...,-,6,online_cracking,vpn,2022-01-18 12:39:45.911967993,2022-01-18 12:41:22.164308071,4,304785.042598,0,4
384804,f62685,1.642510e+09,1.642510e+09,5.046730,172.17.130.196,36416,172.17.130.37,443,tcp,ssl,...,-,6,online_cracking,webserver,2022-01-18 12:41:45.146390915,2022-01-18 12:41:50.193120956,4,304904.277021,0,4
385095,f62748,1.642510e+09,1.642510e+09,6.008968,172.17.130.196,36428,172.17.130.37,443,tcp,ssl,...,-,6,online_cracking,webserver,2022-01-18 12:46:23.429790974,2022-01-18 12:46:29.438759089,4,305182.560421,0,4


In [21]:
df_fp_4 = false_positives(mis_df, 4)
df_fp_4

,flow_id,start_time,end_time,duration,src_ip,sport,dst_ip,dport,proto,service,...,tunnel_parents,ip_proto,label,sensor_host,start_time_dt,end_time_dt,phase,t_rel,y_pred,y_true
384606,f237506,1.642509e+09,1.642509e+09,0.141069,10.9.0.18,35154,10.35.35.206,443,tcp,ssl,...,-,6,benign,attacker0,2022-01-18 12:38:08.034264088,2022-01-18 12:38:08.175333023,0,304687.164894,4,0
384654,f237526,1.642510e+09,1.642510e+09,0.112542,10.9.0.18,35174,10.35.35.206,443,tcp,ssl,...,-,6,benign,attacker0,2022-01-18 12:38:27.805035114,2022-01-18 12:38:27.917577028,0,304706.935665,4,0
384680,f62661,1.642510e+09,1.642510e+09,111.182332,172.17.130.37,35990,172.17.131.81,443,tcp,ssl,...,-,6,proxy,webserver,2022-01-18 12:38:36.407233953,2022-01-18 12:40:27.589565992,0,304715.537864,4,0
384682,f62642,1.642510e+09,1.642510e+09,0.000042,10.9.0.6,36406,172.17.130.37,443,tcp,-,...,-,6,benign,webserver,2022-01-18 12:38:41.578493118,2022-01-18 12:38:41.578535080,0,304720.709123,4,0
384690,f62662,1.642510e+09,1.642510e+09,100.606394,172.17.130.37,35994,172.17.131.81,443,tcp,ssl,...,-,6,proxy,webserver,2022-01-18 12:38:46.985174894,2022-01-18 12:40:27.591568947,0,304726.115805,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
470444,f44910,1.642630e+09,1.642630e+09,5.173146,172.17.130.37,59676,172.17.131.81,443,tcp,ssl,...,-,6,proxy,mail,2022-01-19 22:11:44.248986959,2022-01-19 22:11:49.422132969,0,425503.379617,4,0
470445,f85659,1.642630e+09,1.642630e+09,5.172973,172.17.130.37,59676,172.17.131.81,443,tcp,ssl,...,-,6,proxy,webserver,2022-01-19 22:11:44.249099016,2022-01-19 22:11:49.422071934,0,425503.379729,4,0
470447,f85660,1.642630e+09,1.642630e+09,5.169627,172.17.130.37,59678,172.17.131.81,443,tcp,ssl,...,-,6,proxy,webserver,2022-01-19 22:11:44.252429962,2022-01-19 22:11:49.422056913,0,425503.383060,4,0
470448,f44912,1.642630e+09,1.642630e+09,5.032542,172.17.130.37,59680,172.17.131.81,443,tcp,ssl,...,-,6,proxy,mail,2022-01-19 22:11:44.252965926,2022-01-19 22:11:49.285507917,0,425503.383596,4,0


In [24]:
df_fp_4_src_ips = df_fp_4["src_ip"].value_counts()
print("False Positives for Phase 4 Source IP Distribution:")
print(df_fp_4_src_ips)

False Positives for Phase 4 Source IP Distribution:
src_ip
172.17.130.37      621
172.17.130.196     371
10.35.32.164       158
10.35.35.118       107
10.35.33.116        32
10.9.0.18           23
10.9.0.22           14
10.35.33.144        14
10.9.0.6             3
10.9.0.10            3
192.168.128.4        1
192.168.128.107      1
Name: count, dtype: int64


In [22]:
df_fp_4["dport"].value_counts()

dport
443    1267
80       81
Name: count, dtype: int64

#### Phase 5

In [23]:
# df_fn_5 = false_negatives(mis_df, 5)
# df_fn_5